# 4. (Optional) A neural language model

The models so far count things: words, lemmas, letters. A *neural language model* like BERT reads a whole verse and turns it into a list of 768 numbers, an **embedding**, in which verses the model "reads" as similar sit close together. Here we use **CAMeLBERT-CA**, a BERT model trained by NYU Abu Dhabi's CAMeL Lab on Classical Arabic texts.

**Setup.** This needs extra software (about 1.5 GB):

```
uv sync --extra transformers
```

The first run downloads the model and computes embeddings for all 6,236 verses, which takes several minutes on a laptop. The result is saved in `data/cache/`.

**A caveat to keep in mind.** CAMeLBERT-CA was trained on the OpenITI corpus of premodern Arabic books, which includes Qur'an commentaries and works on the "occasions of revelation" that discuss which verses are Meccan and which Medinan. The model may have absorbed some of the tradition we are trying to test. Large chat models like ChatGPT or Claude have this problem in an extreme form: they have read the traditional labels many times, so asking them to "classify" a surah tests their memory, not the text.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from surahs import load_verses
from surahs.embeddings import verse_embeddings
from surahs.models import cross_validate, make_model

verses = load_verses()
X = verse_embeddings(verses)
X.shape

(6236, 768)

In [2]:
emb_model = make_pipeline(StandardScaler(), LogisticRegression(C=0.05, class_weight="balanced", max_iter=5000))
rows = []
for name, model, feats in [("length only", make_model("length"), None),
                           ("lemmas", make_model("lemmas"), None),
                           ("CAMeLBERT-CA", emb_model, X)]:
    r = cross_validate(model, verses, "cairo_place", X=feats, seeds=(0,), name=name)
    rows.append({"model": name, **{f"verse {k}": v for k, v in r.verse_metrics.items() if k in ("balanced_accuracy", "roc_auc")},
                 **{f"surah {k}": v for k, v in r.surah_metrics.items() if k in ("balanced_accuracy", "roc_auc")}})
pd.DataFrame(rows).round(3)

,model,verse balanced_accuracy,verse roc_auc,surah balanced_accuracy,surah roc_auc
0,length only,0.701,0.772,0.789,0.904
1,lemmas,0.786,0.846,0.917,0.906
2,CAMeLBERT-CA,0.821,0.888,0.923,0.958


**Question 4.1.** Is the neural model worth its cost here? Consider accuracy, but also how easy it is to explain *why* it made a prediction, and the training-data caveat above.

**Question 4.2.** A neural model's embedding of a verse surely encodes its length too. Design an experiment to check how much of the neural model's success is "just length". (Hint: compare it with the length-only model on verses of similar length.)